In [1]:
import pandas as pd 
df = pd.read_excel('/Users/egorilin/Desktop/COMA/2026_04_14_DataSet_Final_MERGED_FILTERED.xlsx')
smiles = df['SMILES']
smiles

0        C1=CC(=CC=C1N/C(=N/C(=NCCCCCCN=C(/N=C(/NC2=CC=...
1        C1=CC(=CC=C1N/C(=N/C(=NCCCCCCN=C(/N=C(/NC2=CC=...
2        C1=CC(=CC=C1N/C(=N/C(=NCCCCCCN=C(/N=C(/NC2=CC=...
3        C1=CC(=CC=C1N/C(=N/C(=NCCCCCCN=C(/N=C(/NC2=CC=...
4        C1=CC(=CC=C1N/C(=N/C(=NCCCCCCN=C(/N=C(/NC2=CC=...
                               ...                        
10959                                                  NaN
10960                                                  NaN
10961                                                  NaN
10962                                                  NaN
10963                                                  NaN
Name: SMILES, Length: 10964, dtype: object

In [2]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = ["Cn1c(=O)c2c(ncn2C)n(C)c1=O", "CC(=O)Oc1ccccc1C(=O)O"]
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

tensor([[-0.4407,  0.3902,  0.7989,  ..., -0.6081, -0.0200,  0.0103],
        [ 0.5943,  0.4527,  0.3437,  ...,  0.1520, -0.3464,  0.5590]])

In [3]:
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

tensor([[-0.4407,  0.3902,  0.7989,  ..., -0.6081, -0.0200,  0.0103],
        [ 0.5943,  0.4527,  0.3437,  ...,  0.1520, -0.3464,  0.5590]])

In [5]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

# Загрузка модели и токенизатора
model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

# Загрузка данных
data = pd.read_excel("/Users/egorilin/Desktop/COMA/2026_04_14_DataSet_Final_MERGED_FILTERED.xlsx")

# Очистка SMILES: удаляем NaN и пустые строки
smiles_series = data["SMILES"].dropna().astype(str)
smiles_series = smiles_series[smiles_series.str.strip() != ""]
smiles_series = smiles_series[smiles_series.str.lower() != "nan"]

# Создаём маску: какие строки остались после очистки
valid_mask = data["SMILES"].isin(smiles_series)
data_clean = data[valid_mask].copy().reset_index(drop=True)
smiles_list = smiles_series.tolist()

print(f"Исходных строк: {len(data)}")
print(f"После очистки SMILES: {len(data_clean)}")
print(f"SMILES для обработки: {len(smiles_list)}")

# Получение эмбеддингов батчами (чтобы не перегрузить память)
batch_size = 32
all_embeddings = []

for i in tqdm(range(0, len(smiles_list), batch_size), desc="Обработка батчей"):
    batch = smiles_list[i:i + batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    all_embeddings.append(outputs.pooler_output)

# Объединяем все батчи
embeddings = torch.cat(all_embeddings, dim=0)  # shape: (N, 768)
print(f"Размер эмбеддингов: {embeddings.shape}")

# Добавляем колонки Molformer_* в очищенный датафрейм
n_features = embeddings.shape[1]
molformer_cols = [f'Molformer_{j}' for j in range(n_features)]

# Конвертируем тензор в numpy и добавляем в DataFrame
embeddings_np = embeddings.cpu().numpy()

# Создаём DataFrame с эмбеддингами и соединяем
embeddings_df = pd.DataFrame(embeddings_np, columns=molformer_cols)
data_with_embeddings = pd.concat([data_clean.reset_index(drop=True), embeddings_df], axis=1)

print(f"Итоговый DataFrame: {data_with_embeddings.shape}")
print(f"Добавлено колонок: {n_features}")

# Сохраняем результат
output_path = "/Users/egorilin/Desktop/COMA/2026_04_14_DataSet_Final_MERGED_FILTERED_Molformer.xlsx"
data_with_embeddings.to_excel(output_path, index=False)
print(f"Сохранено: {output_path}")

Исходных строк: 10964
После очистки SMILES: 10101
SMILES для обработки: 10101


Обработка батчей: 100%|██████████| 316/316 [01:27<00:00,  3.60it/s]


Размер эмбеддингов: torch.Size([10101, 768])
Итоговый DataFrame: (10101, 806)
Добавлено колонок: 768
Сохранено: /Users/egorilin/Desktop/COMA/2026_04_14_DataSet_Final_MERGED_FILTERED_Molformer.xlsx


# Make embeddings on multiregression task

In [7]:
for i in data_with_embeddings.columns:
    print(i)

Name
CODE
Molecular formula
Type
SMILES
Canonical SMILES
Mw, g*mol-1
Lipophilicity (LogP, CLogP, Rm etc.) 
Method 
MIC, ppm
Statistics (±)
MIC, µM
Statistics (±).1
MBC, µM
Statistics (±).2
MIC, mg/L
Statistics (±).3
MBC, mg/L
Statistics (±).4
MBIC, mg/L
Statistics (±).5
MBEC,  mg/L
Statistics (±).6
MBEC, µM
Statistics (±).7
Strain
Incubation
Method
Standard
Solvent
Medium
Detection
Reference (First author, Year, Volume, Issue, Pages)
Year
doi
Corresponding author(s)
Annotation_final
Notes
Molformer_0
Molformer_1
Molformer_2
Molformer_3
Molformer_4
Molformer_5
Molformer_6
Molformer_7
Molformer_8
Molformer_9
Molformer_10
Molformer_11
Molformer_12
Molformer_13
Molformer_14
Molformer_15
Molformer_16
Molformer_17
Molformer_18
Molformer_19
Molformer_20
Molformer_21
Molformer_22
Molformer_23
Molformer_24
Molformer_25
Molformer_26
Molformer_27
Molformer_28
Molformer_29
Molformer_30
Molformer_31
Molformer_32
Molformer_33
Molformer_34
Molformer_35
Molformer_36
Molformer_37
Molformer_38
Molformer